In [1]:
from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {
        'width': 1920,
        'height': 1080,
        'scroll': True,
})

{'width': 1920, 'height': 1080, 'scroll': True}

# Week 06: Monday, AST 5011: Astrophysical Systems

## Stellar Modelling and Astroseismology

### Michael Coughlin <cough052@umn.edu>

With contributions totally ripped off from Carl Fields (UA), Mike Zingale (SUNY), Cole Miller (UMD), and Abi Nolan (Purdue).


## The Equations of Stellar Structure

We begin with the need to construct a model that describes the pressure, density, temperature and luminosity and their derivatives:

$$
\Large{
P = P(\rho,T,\bf{X}) \\
E = E(\rho,T,\bf{X}) \\
\kappa = \kappa(\rho,T,\bf{X}) \\
\epsilon = \epsilon(\rho,T,\bf{X})
}
$$

where $\bf{X}$ is shorthand for composition (as in a specification of nuclear species).

![stellar structure](figures/stellar_structure.jpg)

We also have our various $\nabla$ relations leading to the creation of a fourth-order differential equation in space or mass requiring 4 boundary conditions. 

> How to solve these equations is not trival. But we can make some simplifications that lead to the use of Polytropes. **Polytropes** are pseudo-stellar models for which power law equations of pressure versus density are assumed a priori but where no reference to heat transfer or thermal balance is made.

## Polytropic Equations of State and Polytropes

A polytropic stellar model is defined as

$$
\large
P(r) = K \rho^{1+1/2}(r)
$$

where $n$ is the _polytropic index_ and K is a constant of proportionality. 

From Eqn 7.16-7.26 lead to the _Lane-Emden_ equation, 


a dimensionless form of Poisson's equation where $\xi$ is a dimensionless radius and $\theta$ relates to the density.


Models corresponding to solutions of this equation for a chosen $n$ are called “polytropes of index n” and the solutions themselves are “Lane–Emden solutions” and are denoted by $\theta_{n}(\xi)$.

Solutions to the LE equation are often restricted to $0\lesssim n \lesssim 5$.

### Some relevant examples for Polytropes


1. The pressure of the completely degenerate but nonrelativistic electron gas goes as $\rho^{5/3}$. Hence, by the definition of the polytropic equation of state (7.16), $n$ for this case is 1.5 (or “a three-halves polytrope”).

2. The density exponent for the fully relativistic case is 4/3 and thus $n = 3$ (or “an n equal three polytrope”).

3. Recall that $P\propto\rho^{5/3}$ in an ideal gas convection zone. If no ionization is taking place (almost a contradiction for a real convection zone) then $\Gamma_{2} = 5/3$ and $n = 3/2$ again.

> These represent common scenarios for polytropes and rely on the conditions to which we are comparing.


## Solving the Stellar Structure Equations

Thorough discussion in [MESA 1 Section 6](https://iopscience.iop.org/article/10.1088/0067-0049/220/1/15) for more details. 


## In-Class Exercise: Lane-Emden Solutions

### Goal
Build intuition for how the polytropic index $n$ affects stellar density structure by solving the Lane-Emden equation numerically.

### Task

1. Complete the right-hand side of the Lane-Emden equation.
2. Solve for $n = 0, 1, 1.5, 3, 5$ and plot $\theta(\xi)$ for each.
3. Overplot the analytic solutions for $n = 0$ and $n = 1$:
   - $n = 0$: $\theta = 1 - \xi^2/6$, with $\xi_1 = \sqrt{6}$
   - $n = 1$: $\theta = \sin(\xi)/\xi$, with $\xi_1 = \pi$
4. Which polytrope has the most centrally concentrated density profile?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def lane_emden_rhs(xi, y, n):
    theta, dtheta = y
    if xi < 1e-10:
        return [dtheta, -1.0/3.0]
    dthetadxi = ... # FILL IN
    return [dtheta, dthetadxi]

fig, ax = plt.subplots(figsize=(8, 5))

for n in [0, 1, 1.5, 3, 5]:
    xi_end = 30.0
    sol = solve_ivp(lane_emden_rhs, [1e-6, xi_end],
                    [1 - 1e-6**2/6, -1e-6/3],
                    args=(n,), max_step=0.01, dense_output=True)

    xi = sol.t
    theta = np.clip(sol.y[0], 0, None)

    # Find xi_1 (first zero crossing)
    idx = np.where(theta <= 0)[0]
    if len(idx) > 0:
        mask = np.arange(idx[0])
    else:
        mask = np.arange(len(xi))

    ax.plot(xi[mask], theta[mask], label=f'n={n}')

# Analytic: n=0
xi_an0 = np.linspace(0.01, np.sqrt(6), 200)
ax.plot(xi_an0, ... # FILL IN: n=0 analytic
        , 'k--', alpha=0.5, label='n=0 analytic')

# Analytic: n=1
xi_an1 = np.linspace(0.01, np.pi, 200)
ax.plot(xi_an1, ... # FILL IN: n=1 analytic
        , 'k:', alpha=0.5, label='n=1 analytic')

ax.set_xlabel(r'$\xi$')
ax.set_ylabel(r'$\theta(\xi)$')
ax.set_title('Lane-Emden Solutions')
ax.legend()
ax.set_xlim(0, 12)
ax.set_ylim(-0.1, 1.1)
plt.show()

<details>
<summary>Solution</summary>

```python
    dthetadxi = -2.0/xi * dtheta - theta**n if theta > 0 else 0

    # n=0 analytic
    1 - xi_an0**2 / 6

    # n=1 analytic
    np.sin(xi_an1) / xi_an1
```

Higher $n$ produces a more centrally concentrated density profile. The $n = 5$ polytrope has infinite radius ($\xi_1 \to \infty$) and an extremely sharp central peak. The $n = 0$ polytrope has uniform density. Physical stellar models typically fall in the range $n = 1.5$ to $3$.

</details>

## Radial Stellar Pulsations



Stellar pulsations start from _pressure waves_ (sound waves that resonate in the stellar interior). 

These radial oscillations act like standing waves with a node at the center and open end at the surface. 

**Fundamental mode** - one node at the center

**First overtone** - one node at the center and one between center and surface

**Second overtone** - one node at the center and two between center and surface

Most radially pulsating stars, _Cephieds_, are oscillating in their fundemental mode. 


### Adiabatic Oscillations

In practice, oscillations will be a about some equilibrium structure at equilibrium radius $r_{0}$ and mass $m$. Under these assumptions, the radial oscillation frequency is found to be: 

$$
\omega^{2} = (3\gamma_{\rm{ad}}-4)\frac{Gm}{r^{3}_{0}}
$$

Recall $\gamma_{\rm{ad}}\equiv\Gamma_{1}$.

For $\gamma_{\rm{ad}}>4/3$, we have $\omega^{2}>0$ and **dynamical stability**.

For $\gamma_{\rm{ad}}<4/3$, we have $\omega^{2}<0$, indicating exponential growth of the perturbations and **dynamical instability**.


If we average $\omega$ over the entire star, we can get an estimate for the **pulsation frequency of the fundamental mode**:

$$
\Pi_{0} = \frac{2\pi}{\sqrt{(3\gamma_{\rm{ad}}-4)}\frac{GM}{R^3}} \equiv \left ( \frac{3\pi}{(3\gamma_{\rm{ad}}-4)G\bar{\rho}} \right )^{1/2}
$$

where we have used the total radius and mass of the star. 

We can generalize this to:

$$
\Pi = Q \left ( \frac{\bar{\rho}}{\rho_{\odot}} \right )
$$

We have $Q\approx0.04 \ \rm{days}$ for the fundamental mode and smaller for higher overtones.

## In-Class Exercise: Pulsation Period from Mean Density

### Goal
Use the period–mean density relation to estimate fundamental pulsation periods for different types of variable stars.

### Task

1. Compute the mean density $\bar{\rho} = 3M / (4\pi R^3)$ for each star.
2. Compute the fundamental period $\Pi_0 = \left(\frac{3\pi}{(3\gamma_{\rm ad} - 4) G \bar{\rho}}\right)^{1/2}$ assuming $\gamma_{\rm ad} = 5/3$.
3. Plot period versus mean density.
4. Do the computed periods match the observed ranges?

| Star type | Observed period range |
|-----------|---------------------|
| $\delta$ Scuti | 0.02 – 0.25 days |
| RR Lyrae | 0.2 – 1.0 days |
| Cepheid | 1 – 100 days |
| Mira | 100 – 700 days |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Constants (cgs)
G = 6.67e-8
Msun = 1.989e33
Rsun = 6.96e10

# Stellar parameters [M/Msun, R/Rsun]
stars = {
    'delta Scuti': {'M': 2.0, 'R': 2.5},
    'RR Lyrae':    {'M': 0.7, 'R': 5.0},
    'Cepheid':     {'M': 5.0, 'R': 40.0},
    'Mira':        {'M': 1.0, 'R': 300.0},
}

gamma_ad = 5.0 / 3.0

names = []
periods = []
rho_bars = []

for name, params in stars.items():
    M = params['M'] * Msun
    R = params['R'] * Rsun

    rho_bar = ... # FILL IN: mean density
    Pi_0 = ... # FILL IN: fundamental period in seconds

    names.append(name)
    periods.append(Pi_0 / 86400)  # convert to days
    rho_bars.append(rho_bar)
    print(f"{name}: rho_bar = {rho_bar:.3e} g/cm^3, Pi_0 = {Pi_0/86400:.2f} days")

# Plot period vs mean density
plt.figure(figsize=(6, 4))
plt.scatter(rho_bars, periods, zorder=5)
for i, name in enumerate(names):
    plt.annotate(name, (rho_bars[i], periods[i]),
                 textcoords="offset points", xytext=(5, 5))
plt.xlabel(r'Mean density $\bar{\rho}$ [g/cm$^3$]')
plt.ylabel('Fundamental period [days]')
plt.xscale('log')
plt.yscale('log')
plt.title('Period\u2013Mean Density Relation')
plt.show()

<details>
<summary>Solution</summary>

```python
    rho_bar = 3 * M / (4 * np.pi * R**3)
    Pi_0 = np.sqrt(3 * np.pi / ((3 * gamma_ad - 4) * G * rho_bar))
```

The computed periods should fall within the observed ranges listed above. Lower mean density corresponds to longer pulsation periods. This simple estimate works surprisingly well because the period–mean density relation is robust across different stellar types.

</details>

### Driving and damping of pulsations

**$\epsilon$-mechanism**:

If the compressed region occurs where nuclear reactions occur, $T$ rises and thus the nuclear energy generation rate. This could satisfy the criterion to grow the pulsation. Always taking place in the cores, but very small amplitudes. Might be important for very massive stars, not so for Cepheids. 


**$\kappa$-mechanism**: 

If the compressed layer becomes more opaque, heat is trapped. This causes and increase in $P$ and $T$, which drive expansion and release of the heat as a result of the reduction of opacity. This can maintain the pulsation cycle and drive large scale pulsations. 

We can write a requirement for this mechanism as:

$$
\left ( \frac{d \ \textrm{ln} \ \kappa}{d \ \textrm{ln} \ P} \right )_{\rm{ad}} = \left ( \frac{\partial \ \textrm{ln} \ \kappa}{\partial \ \textrm{ln} \ P} \right )_{T} +  \left ( \frac{\partial \ \textrm{ln} \ \kappa}{\partial \ \textrm{ln} \ T} \right )_{P} \nabla_{\rm{ad}} \equiv \kappa_{P} + \kappa_{T} \nabla_{\rm{ad}}
$$
for pulsations via this mechanism we require $\kappa_{P} + \kappa_{T} \nabla_{\rm{ad}}\gt0$.


### Pathways to Instability

1) **$\kappa_{T}>0$**, occurs when $H^{-}$ opacity dominates in very cool stars with $T\lt10^{4}(\rm{K})$. Cepheids are too hot for this. 

2) Small $\nabla_{\rm{ad}}$ for a Kramers like opacity. We found that $\nabla_{\rm{ad}}$ can be reduced in _partial ionization zones_. 

    **Two key partial ionization zones**
    - $T\approx1.5\times10^{4}$ - where $H\rightarrow H^{+}+e^{-}$ and $He\rightarrow He^{+}+e^{-}$ can occur. 
    - $T\approx4\times10^{4}$ - where helium becomes doubly ionized $He^{+}\rightarrow He^{++}+e^{-}$ (more opaque)
    
These considerations determine the location of the instability strip.

## In-Class Exercise: Kappa-Mechanism Driving Condition

### Goal
Explore when the opacity-driven pulsation condition is satisfied and understand why partial ionization zones are essential.

### Background

For Kramers opacity $\kappa \sim \rho \, T^{-3.5}$, using an ideal gas ($P \propto \rho T$):
- At constant $T$: $\rho \propto P$, so $\kappa \propto P \cdot T^{-3.5}$, giving $\kappa_P = 1$
- At constant $P$: $\rho \propto 1/T$, so $\kappa \propto T^{-4.5}$, giving $\kappa_T = -4.5$

The driving condition is $\kappa_P + \kappa_T \nabla_{\rm ad} > 0$.

### Task

1. Evaluate the driving condition as a function of $\nabla_{\rm ad}$.
2. Find the critical $\nabla_{\rm ad}$ below which driving occurs.
3. Mark where the ideal gas value ($\nabla_{\rm ad} = 0.4$) falls.
4. Is driving possible in normal stellar material? What about partial ionization zones?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Kramers opacity derivatives
kappa_P = 1.0
kappa_T = -4.5

# Range of nabla_ad
nabla_ad = np.linspace(0.0, 0.5, 100)

# Driving condition
driving = ... # FILL IN

plt.figure(figsize=(6, 4))
plt.plot(nabla_ad, driving, 'b-', linewidth=2)
plt.axhline(0, color='k', linestyle='--', alpha=0.5)
plt.axvline(0.4, color='r', linestyle=':', label=r'Ideal gas $\nabla_{\rm ad} = 0.4$')
plt.fill_between(nabla_ad, driving, 0,
                 where=(driving > 0), alpha=0.3, color='green', label='Driving')
plt.fill_between(nabla_ad, driving, 0,
                 where=(driving < 0), alpha=0.3, color='red', label='Damping')
plt.xlabel(r'$\nabla_{\rm ad}$')
plt.ylabel(r'$\kappa_P + \kappa_T \nabla_{\rm ad}$')
plt.title(r'$\kappa$-Mechanism Driving Condition (Kramers Opacity)')
plt.legend()
plt.show()

# Critical value
nabla_crit = ... # FILL IN
print(f"Driving requires nabla_ad < {nabla_crit:.3f}")
print(f"Ideal gas nabla_ad = 0.4: condition = {kappa_P + kappa_T * 0.4:.2f} (damped)")
print(f"Partial ionization nabla_ad ~ 0.1: condition = {kappa_P + kappa_T * 0.1:.2f} (driving!)")

<details>
<summary>Solution</summary>

```python
driving = kappa_P + kappa_T * nabla_ad

nabla_crit = -kappa_P / kappa_T
```

For normal stellar material with $\nabla_{\rm ad} = 0.4$, the driving condition gives $1 + (-4.5)(0.4) = -0.8 < 0$, so pulsations are **damped**.

In partial ionization zones, $\nabla_{\rm ad}$ drops well below 0.222 (e.g., to $\sim 0.1$), making $1 + (-4.5)(0.1) = 0.55 > 0$: pulsations are **driven**.

This is why the instability strip exists at specific temperatures — it corresponds to where partial ionization zones sit at the right depth in the stellar envelope.

</details>

**Example: Cepheids**



The main gas involved is thought to be helium. The cycle is driven by the fact doubly ionized helium, the form adopted at high temperatures, is more opaque than singly ionized helium. 

1. Outer layer is compressed
2. This layer is heated, helium is doubly ionized
3. The increased heat absorbs enough heat to expand
4. Cools and Helium is back to singly ionized (less opaque)

Cepheid variables become dimmest during the part of the cycle when the helium is doubly ionized.

![polaris](figures/polaris.gif)

A series of images of the pole star, Polaris, which is a Cepheid type variable. 4 frames taken at 24 hour intervals covering Polaris' approximately 4 day cycle during which its brightness varies by 0.27 magnitudes. **Credit: (Tim Wetherell 2022)**


### Modern Pulsation Codes

**GYRE** by R. H. D. Townsend, S. A. Teitler - an open-source stellar oscillation code based on a new Magnus Multiple Shooting scheme. Paper [here](https://academic.oup.com/mnras/article/435/4/3406/1033475). Solves the stellar pulsation equations (both adiabatic and non-adiabatic) using a novel Magnus Multiple Shooting numerical scheme. Integrated into MESA since MESA Paper II. 


**RSP** MESA [module](https://iopscience.iop.org/article/10.3847/1538-4365/ab2241/pdf) - model the time evolution of large amplitude, self-excited, nonlinear pulsations over many cycles to produce luminosity and radial velocity histories that can be compared to observations.

# In-Class Assignment: Comparing Polytropes to MESA models

### Learning Objectives

- explore the relationship between lane-emden solutions and physical variables
- identify physics informed polytrope profiles for models of massive stars and white dwarfs

# Lane-Emden polytrope equation

## Derivation

We consider just the equations of hydrostatic equilibrium and mass continuity from our
equations of stellar structure.

$$\frac{dP}{dr} = -\rho \frac{GM(r)}{r^2}$$

$$\frac{dM}{dr} = 4\pi r^2 \rho$$

We can close the system via an equation of state of the form:
$P(\rho)$, which we write as:

$$P = K \rho^{1+1/n}$$

where $n$ is called the _polytropic index_.

With a bit of algebra, we can combine these equations into a single second-order ODE for density:

$$\left ( \frac{n+1}{n} \right ) \frac{K}{4\pi G} \frac{1}{r^2} \frac{d}{dr} \left ( \frac{r^2}{\rho^{(n-1)/n}} \frac{d\rho}{dr} \right ) = -\rho$$

and then make it dimensionless, but expressing the density in terms of the central density, $\rho_c$:

$$\rho(r) = \rho_c \theta^n(r)$$

where we note that $0 \le \theta \le 1$, and a lenghtscale $\alpha$ such that $r = \alpha \xi$:

$$\alpha^2 = \frac{(n+1)P_c}{4\pi G\rho_c^2}$$

Giving

$$\frac{1}{\xi^2}\frac{d}{d\xi} \left ( \xi^2 \frac{d\theta}{d\xi} \right ) = -\theta^n$$

The boundary conditions are:
* $\theta(\xi=0) = 1$
* $d\theta / d\xi |_{\xi=0} = 0$ ($g = 0$ at the center, so $dP/dr = d\rho/dr = 0$)

## System of first order equations

We will rewrite this as 2 first order equations, taking $y = \theta$, $z = \theta^\prime$:

$$
\begin{align}
\frac{dy}{d\xi} &=& z \\
\frac{dz}{d\xi} &=& -\frac{2}{\xi}z - y^n
\end{align}
$$

Then we have:

* $y(0) = 1$
* $z(0) = 0$

Notice that if we evaluate the system at $\xi = 0$, then the $dz/d\xi$ term blows up.  We need to use an expansion there.

In the limit $\xi \rightarrow 0$, the solution takes the form:

$$\theta(\xi) = 1 - \frac{1}{6}\xi^2 + \frac{n}{120} \xi^4 + \ldots$$

which allows us to simply the $dz/d\xi$ near $\xi = 0$ as:

$$\frac{dz}{d\xi} \approx -\frac{1}{3}$$


## Stopping point

But we have the problem that we do not know the stopping point, $\xi_1$.

We estimate the radius, $\xi_1$ each step and make sure that the next step does not take us past that estimate.  This prevents us from having negative $\theta$ values.  Given a point $(\xi_p, \theta(\xi_p))$, and the derivative at that point, $\theta^\prime_p = d\theta/d\xi |_{\xi_p}$, we can write the equation of a line as:

$$
    \theta(\xi) - \theta_p = \theta^\prime_p (\xi - \xi_p)
$$


Then we can ask when does $\theta$ become zero, finding:

$$
    \xi = -\frac{\theta_p}{\theta^\prime_p} + \xi_p
$$

or in terms of $y$ and $z$, 

$$
    \xi = -\frac{y_p}{z_p} + \xi_p
$$

This is our estimate of $\xi_1$.  We then make sure our stepsize $h$ is small enough that we do not go beyond this estimate.

## Implementation

This is our main class that does the integration.  We initialize it with the polytopic index, and then it will integrate the system for us.  We can then plot it or get
the parameters $\xi_1$ and $-\xi_1^2 d\theta/d\xi |_{\xi_1}$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class Polytrope:
    """a polytrope of index n"""
    def __init__(self, n, h0=1.e-2, tol=1.e-12):
        self.n = n
        self.xi = []
        self.theta = []
        self.dtheta_dxi = []
        
        self._integrate(h0, tol)

    def _integrate(self, h0, tol):
        """integrate the Lane-Emden system"""

        # our solution vector q = (y, z)
        q = np.zeros(2, dtype=np.float64)
        xi = 0.0

        h = h0

        # initial conditions
        q[0] = 1.0
        q[1] = 0.0

        while h > tol:
            # 4th order RK integration -- first find the slopes
            k1 = self._rhs(xi, q)
            k2 = self._rhs(xi+0.5*h, q+0.5*h*k1)
            k3 = self._rhs(xi+0.5*h, q+0.5*h*k2)
            k4 = self._rhs(xi+h, q+h*k3)

            # now update the solution to the new xi
            q += (h/6.0)*(k1 + 2*k2 + 2*k3 + k4)
            xi += h

            # set the new stepsize--our systems is always convex
            # (theta'' < 0), so the intersection of theta' with the
            # x-axis will always be a conservative estimate of the
            # radius of the star.  Make sure that the stepsize does
            # not take us past that.
            R_est = xi - q[0]/q[1]

            if xi + h > R_est:
                h = -q[0]/q[1]

            # store the solution:
            self.xi.append(xi)
            self.theta.append(q[0])
            self.dtheta_dxi.append(q[1])

        self.xi = np.array(self.xi)
        self.theta = np.array(self.theta)
        self.dtheta_dxi = np.array(self.dtheta_dxi)

    def _rhs(self, xi, q):
        """ the righthand side of the LE system, q' = f"""

        f = np.zeros_like(q)

        # y' = z
        f[0] = q[1]
        
        # for z', we need to use the expansion if we are at xi = 0,
        # to avoid dividing by 0
        if xi == 0.0:
            f[1] = (2.0/3.0) - q[0]**self.n
        else:
            f[1] = -2.0*q[1]/xi - q[0]**self.n

        return f

    def get_params(self):
        """ return the standard polytrope parameters xi_1,
        and [-xi**2 theta']_{xi_1} """
        xi1 = self.xi[-1]
        p2 = -xi1**2 * self.dtheta_dxi[-1]
        return xi1, p2

    def plot(self):
        """ plot the solution """
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.plot(self.xi, self.theta, label=r"$\theta$")
        ax.plot(self.xi, self.theta**self.n, label=r"$\rho/\rho_c$")
        ax.set_xlabel(r"$\xi$")
        ax.legend(frameon=False)
        return fig

## a. 

Plot the results of an $n=4$ polytrope.

In [ ]:
## a result here

From the readings in HKT 7.1-7.2. We found some of the following relations:

Equation 7.20-7.22:


$$
\Large
\rho(r) = \rho_{\rm{c}}\theta^{n}(r) 
$$

$$
\Large
P(r) =  P_{\rm{c}} \theta^{1+n}(r)
$$

$$
\Large
P_{\rm{c}} =  K \rho^{1+1/n}_{\rm{c}} 
$$

and 

Equation 7.25:

$$
\Large
r^{2}_{n} = \frac{(n+1)P_{c}}{4\pi G \rho^{2}_{c}}
$$

and 

$$
\Large
r = r_{n} \xi
$$

## b.

Download the following model files locally. These data were produced using the `make_co_wd` test suite. 

* $0.6 M_{\odot}$ WD: [co_wd_profile.data](data/week6/co_wd_profile.data);


Then, 

1. produce a polytrope of index `n` 
2. compute `K` using the central values from the MESA model (for pandas you can use `iloc`).
3. compute `r_n` using the central values from the MESA model 
3. compute the pressure and density profiles from the above constants and the polytrope solutions 
4. plot them on the same plot as the MESA model - one plot for pressue one for density
5. try different values of `n` until a best fit is obtained

In [ ]:
## b results here
G = 6.67e-8

## c.

Explain in words, the physics motivating the choice of `n` in comparing to the white dwarf. 

c results here.

# In-Class Assignment: Radially Pulsating Stars - Cephieds

### Learning Objectives

- explore boundaries of instability strip
- verify the dominant $\kappa$-mechanism source for Cepheids
- compare analytical estimates for the period with MESA output
- compute a phase folded lightcurve to determine changes in pulsation over time.

Download the following model files locally. 

* Cephied RSP model: [cephied_history.data](data/cephied_history.data);
                [cephied_profile.data](data/cephied_profile.data);

## a.

Using the history data, 

1. produce an HR diagram of the Cephied data and confirm that it is bounded within the instability strip. 
2. on the same plot, plot the approximate bounds of the instability strip with dashed lines: 
    - blue edge point 1 ($\textrm{log} L_{1}=5.5$,$\textrm{log} T_{\rm{eff}}=3.7$); 
    - blue edge point 2 ($\textrm{log} L_{2}=1.0$,$\textrm{log} T_{\rm{eff}}=3.93$) and 
    - red edge point 1 ($\textrm{log} L_{1}=5.5$,$\textrm{log} T_{\rm{eff}}=3.6$) and 
    - red edge point 2 ($\textrm{log} L_{2}=1.0$,$\textrm{log} T_{\rm{eff}}=3.83$).

In [ ]:
## a results here

## b.

Using the profile data, 

1. verify the dominant ionization sources via the existance of doubly ionized He and singly ionization H using a plot
2. comment on the location of the ionization zone, that is, is it near the high density region (near the red edge of the strip), or low density (blue edge)

3. plot `grada`, does it decrease as expected in the partial ionization zones?

In [ ]:
## b results here

## c.

1. Using Eqn. 8.21 in HKT, make an esimtate of the period in days. Recall that $\left < \rho \right > \approx M/R^{3}$ or Eqn 10.5 in Onno Pol's notes. 

2. Plot the total radius or luminosity over time in days `star_age_in_days`, does the estimated period match by eye?


In [ ]:
## c results here

## d. 

Using the final computed period in the history data `rsp_period_in_days`.

Compute the phase using the period, star age in days, and luminosity. 

1. Produce a "phase-folded" luminosity plot - phase vs luminosity.

$$
\theta = (t \ \% \ P) / P
$$

Next, sort by phase to get the argument indices

```
sort_index = np.argsort(phase)
```

```
phase_sorted = phase[sort_index]
lum_sorted = lum[sort_index]
```

where $\%$ is the modulus operator in python. More details and examples are available [here](https://docs.astropy.org/en/latest/timeseries/analysis.html) including using Astropy to automatically do the folding. 

> Comment briefly on if it is possible to observe the growth of the amplitude with subsequent periods. The answer is also in the history data `growth` for each period for the curious.

In [ ]:
## d result here